# Pipeline -- phase 7

Phases 4-6 all ran inside this kernel: fit, score, log. Stop the app and
none of it reruns. This notebook moves the same work onto SageMaker
compute as a DAG.

```
preprocess -> train -> evaluate -> [rmse gate] -> register
```

1. upsert the pipeline defined in `src/pipeline.py`
2. start a run and watch the steps
3. confirm the metrics match phases 5 and 6
4. find the new version sitting in the registry, pending approval

The definition is in `src/pipeline.py`, not in Terraform. Terraform owns
policy 3 and the model package group -- infrastructure. The DAG changes
when the model changes, so it ships with the training code.

## 1. Setup

`src/` is already on disk -- the lifecycle config cloned this repo into
the space in phase 4. The notebook runs from `notebooks/`, so the
pipeline's `source_dir="src"` needs the repo root as the working
directory.

In [1]:
import os

# ScriptProcessor uploads `code=` relative to cwd, so this must be the
# repo root and not notebooks/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

/home/sagemaker-user/sagemaker


The three values below come from the domain stack:

```
terraform -chdir=infra/domain output -raw data_bucket
terraform -chdir=infra/domain output -raw alice_role_arn
terraform -chdir=infra/domain output -raw model_package_group
```

In [2]:
import boto3

REGION = "ca-central-1"

BUCKET = "sagemaker-domain-dev-data-pqkx2l"
ROLE = "arn:aws:iam::099139718958:role/sagemaker-domain-dev-alice-role"
MODEL_PACKAGE_GROUP = "sagemaker-domain-dev-bike-sharing-rf"

sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

# Fails here rather than three steps into a billed run if policy 3 has
# not been applied yet.
sm.describe_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
print(f"registry group {MODEL_PACKAGE_GROUP} reachable")

registry group sagemaker-domain-dev-bike-sharing-rf reachable


## 2. Build and upsert

`build()` returns the Pipeline object; nothing exists in AWS until
`upsert()`. Upsert rather than create -- rerunning this cell is the
normal way to iterate on a definition.

In [3]:
import sys

# src/ has no __init__.py -- it is a directory of entry-point scripts
# that SageMaker packages and runs, not a package. Put it on the path
# rather than adding one, since `source_dir="src"` ships whatever is in
# there to the container.
sys.path.insert(0, "src")

from pipeline import PIPELINE_NAME, build

pipeline = build(
    bucket=BUCKET,
    role=ROLE,
    model_package_group=MODEL_PACKAGE_GROUP,
    image="341280168497.dkr.ecr.ca-central-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
    instance_type="ml.m5.large",
    rmse_threshold=150.0,
)

pipeline.upsert(role_arn=ROLE)
print(f"upserted {PIPELINE_NAME}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[07/31/26 19:44:38] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=7587061;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=7587062;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


                    INFO     StoppingCondition not provided. Using default:                         ]8;id=7587069;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=7587070;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

...

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7587156;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587157;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[07/31/26 19:44:41] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=7587162;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587163;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7587168;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587169;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=7587174;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587175;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

upserted bike-sharing-rf


The definition is JSON underneath. Worth looking at once -- this is
what the Terraform `aws_sagemaker_pipeline` resource would have needed
hand-written, and what the SDK generated instead.

In [4]:
import json

definition = json.loads(pipeline.definition())

print(f"{len(definition['Steps'])} top-level steps")
for step in definition["Steps"]:
    print(f"  {step['Name']:12} {step['Type']}")

print()
print(f"definition is {len(pipeline.definition()):,} characters of JSON")

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7587180;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587181;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[07/31/26 19:44:42] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=7587186;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587187;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7587192;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587193;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=7587198;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587199;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

4 top-level steps
  Preprocess   Processing
  Train        Training
  Evaluate     Processing
  CheckRmse    Condition



                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7587204;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587205;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[07/31/26 19:44:43] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=7587210;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587211;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7587216;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587217;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=7587222;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7587223;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

definition is 6,281 characters of JSON


## 3. Run it

This is the billed part: three jobs on `ml.m5.large`, a few minutes
each. The instances are provisioned and torn down per step -- nothing
keeps running afterwards, unlike the JupyterLab app.

In [5]:
execution = pipeline.start()

print(execution.arn)

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=7587228;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=7587229;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

arn:aws:sagemaker:ca-central-1:099139718958:pipeline/bike-sharing-rf/execution/k8oram8w1his


In [6]:
# Blocks until the run finishes or fails. Roughly 8-12 minutes cold;
# a rerun that hits the step cache is much faster.
execution.wait()

print(execution.describe()["PipelineExecutionStatus"])

Succeeded


In [7]:
for step in execution.list_steps():
    status = step["StepStatus"]
    cached = " (cached)" if step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn") else ""
    print(f"{step['StepName']:12} {status}{cached}")

Register     Succeeded
CheckRmse    Succeeded
Evaluate     Succeeded
Train        Succeeded
Preprocess   Succeeded


## 4. Cross-check against phases 5 and 6

The pipeline refit the model on different hardware, from the raw CSV,
with no notebook involved. If rmse still matches, the reproducibility
claim now holds across machines -- not just across kernels.

In [8]:
import io

obj = s3.get_object(Bucket=BUCKET, Key="model/evaluation/evaluation.json")
report = json.loads(obj["Body"].read())

metrics = report["regression_metrics"]
rmse = metrics["rmse"]["value"]

print(f"pipeline rmse={rmse:.4f} r2={metrics['r2']['value']:.4f}")
print(f"baseline rmse={metrics['baseline_rmse']['value']:.4f}")

pipeline rmse=126.3496 r2=0.6342
baseline rmse=227.8080


In [9]:
import numpy as np

obj = s3.get_object(Bucket=BUCKET, Key="model/eval.json")
phase5 = json.loads(obj["Body"].read())

recorded = phase5["runs"]["A-leaf5"]["metrics"]["rmse"]

# Not exact: the notebook fit on featured/ as phase 4 wrote it, the
# pipeline re-derived that frame from raw/. Same rows, same seed, so the
# gap should be nil -- but assert a tolerance rather than equality,
# because a float that survived a parquet round trip is not the identity
# check the mlflow notebook could make.
assert np.isclose(rmse, recorded, rtol=1e-3), f"pipeline {rmse} != phase 5 {recorded}"

print(f"{rmse:.4f} matches phase 5's {recorded:.4f}")

126.3496 matches phase 5's 126.3496


## 5. The gate

The condition step is the point of this phase. The run above passed, so
a version was registered -- pending, not approved.

In [10]:
versions = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
)["ModelPackageSummaryList"]

for v in versions:
    print(f"v{v['ModelPackageVersion']:<3} {v['ModelApprovalStatus']:<22} {v['CreationTime']:%Y-%m-%d %H:%M}")

latest = versions[0]
assert latest["ModelApprovalStatus"] == "PendingManualApproval", latest["ModelApprovalStatus"]
print()
print("newest version is pending -- phase 8 will not deploy it until approved")

v1   PendingManualApproval  2026-07-31 19:54

newest version is pending -- phase 8 will not deploy it until approved


### What the gate actually blocks

Worth proving rather than trusting. Start a run with a threshold the
model cannot clear and the register step is skipped -- the run still
ends green, but nothing new lands in the registry.

This costs another few minutes of compute. Skip it if you have seen
enough; the step cache means preprocess and train are reused.

In [11]:
strict = pipeline.start(parameters={"RmseThreshold": 50.0})
strict.wait()

print(strict.describe()["PipelineExecutionStatus"])

steps = {s["StepName"]: s["StepStatus"] for s in strict.list_steps()}
print(steps)

after = sm.list_model_packages(ModelPackageGroupName=MODEL_PACKAGE_GROUP)["ModelPackageSummaryList"]
assert len(after) == len(versions), "a version was registered despite failing the gate"

print()
print(f"run succeeded, register skipped, still {len(after)} version(s)")

[07/31/26 19:55:16] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=7587234;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=7587235;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Succeeded
{'CheckRmse': 'Succeeded', 'Evaluate': 'Succeeded', 'Train': 'Succeeded', 'Preprocess': 'Succeeded'}

run succeeded, register skipped, still 1 version(s)


## 6. Approve

Approval is deliberately a separate act from training. In phase 10 this
is the handoff: bob's run registers a version, alice approves it.

Studio: **Models > Model registry >** the group **>** the version **>
Update status**. Or from here:

In [12]:
sm.update_model_package(
    ModelPackageArn=latest["ModelPackageArn"],
    ModelApprovalStatus="Approved",
)

print(f"v{latest['ModelPackageVersion']} approved -- phase 8 can deploy it")

v1 approved -- phase 8 can deploy it


## 7. What phase 6 registered, and why this is different

Phase 6 called `mlflow.register_model()` by hand after eyeballing two
runs. That path still works and still has its place for exploration --
but it registers whatever the notebook happened to fit, and it needs a
person in the loop to happen at all.

This registers only what cleared the gate, and it reruns unattended.
That is the whole difference between the two phases; the model is the
same one.

The MLflow app's `AutoModelRegistrationEnabled` mode was left off for
the same reason -- it would promote every `log_model` call, which is the
ungated path this phase replaces.

## Cost

Nothing here bills between runs. Each step provisions an instance and
releases it, and the pipeline definition itself is free to store.

The JupyterLab app running this notebook is still the expensive thing in
the stack -- stop it when done.